## Behaviour steering knowledge

This notebook gives nbskill a small, practical memory. A memory item is a regex plus a note: agents can store a repeated good or bad practice once, and later style checks can warn when notebook code matches it.

The lookup is intentionally deterministic. nbskill does not try to infer intent from prose at warning time; it scans the exact notebook or cell being inspected and reports matching rules.

In [ ]:
#| default_exp knowledge

### Imports

The knowledge file is plain JSON. Notebook scanning only needs notebook reads, code-cell source text, and regex matching.

In [ ]:
#| export
import glob,json,os,re,time
from pathlib import Path

from fastcore.nbio import read_nb
from fastcore.script import call_parse

from nbskill.foundation import cell_source, cli_return, tracked_call

### Built-in examples

nbskill ships a few default behaviours so the feature works before a user has saved anything. User memory is merged on top of these defaults, and defaults can be disabled with `NBSKILL_KNOWLEDGE_DEFAULTS=0`.

In [ ]:
#| export
_DEFAULT_BEHAVIOURS = [
    {
        "id": "avoid-aliased-function-imports",
        "regex": r"from\s+\S+\s+import\s+\w+\s+as\s+\w+",
        "note": "Do not import functions by alias unless it is needed to avoid a real conflict.",
        "source": "default",
    },
    {
        "id": "avoid-wildcard-imports",
        "regex": r"from\s+\S+\s+import\s+\*",
        "note": "Avoid wildcard imports; import the names used by the cell.",
        "source": "default",
    },
    {
        "id": "avoid-shell-true",
        "regex": r"subprocess\.(run|Popen|call|check_call|check_output)\([^)]*shell\s*=\s*True",
        "note": "Avoid shell=True unless shell behavior is explicitly required.",
        "source": "default",
    },
    {
        "id": "avoid-eval-exec",
        "regex": r"\b(eval|exec)\s*\(",
        "note": "Avoid eval/exec unless dynamic execution is the actual goal.",
        "source": "default",
    },
    {
        "id": "avoid-destructive-git",
        "regex": r"git\s+(reset\s+--hard|clean\s+-fd|checkout\s+--)",
        "note": "Do not run destructive git cleanup commands unless the user explicitly asked for them.",
        "source": "default",
    },
]


def default_knowledge():
    "Return built-in behaviour steering examples."
    return {"version": 1, "behaviours": [dict(item) for item in _DEFAULT_BEHAVIOURS]}

In [ ]:
defaults = default_knowledge()["behaviours"]
print([item["id"] for item in defaults[:2]])
assert any(item["id"] == "avoid-aliased-function-imports" for item in defaults)

['avoid-aliased-function-imports', 'avoid-wildcard-imports']


### JSON memory

User-added rules live in one JSON file. The default location is `~/.nbskill-knowledge.json`, with an environment override for tests or project-specific memory.

In [ ]:
#| export
def knowledge_path(path=None):
    "Return the JSON file used for behaviour steering memory."
    default = Path.home() / ".nbskill-knowledge.json"
    return Path(path or os.environ.get("NBSKILL_KNOWLEDGE_PATH", default)).expanduser()

In [ ]:
#| export
def _empty_knowledge():
    return {"version": 1, "behaviours": []}

In [ ]:
#| export
def _defaults_enabled():
    return os.environ.get("NBSKILL_KNOWLEDGE_DEFAULTS", "1").lower() not in {"0", "false", "no"}

In [ ]:
#| export
def _merge_behaviours(*groups):
    seen, behaviours = set(), []
    for group in groups:
        for item in group:
            key = item.get("id") or item.get("regex")
            if key in seen: continue
            seen.add(key)
            behaviours.append(dict(item))
    return behaviours

In [ ]:
#| export
def _load_user_knowledge(path=None):
    pth = knowledge_path(path)
    try: data = json.loads(pth.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError, OSError): data = _empty_knowledge()
    data.setdefault("version", 1)
    data.setdefault("behaviours", [])
    return data

In [ ]:
#| export
def load_knowledge(path=None, include_defaults=True):
    "Load behaviour steering rules from defaults and the JSON memory file."
    data = _load_user_knowledge(path)
    if include_defaults and _defaults_enabled():
        data["behaviours"] = _merge_behaviours(default_knowledge()["behaviours"], data.get("behaviours", []))
    return data

In [ ]:
from nbskill.foundation import demo_path, remove_demo_path

memory_path = demo_path("knowledge-empty.json")
try:
    loaded = load_knowledge(memory_path)
    print(loaded["behaviours"][0]["id"])
    assert loaded["behaviours"]
    assert _load_user_knowledge(memory_path)["behaviours"] == []
finally:
    remove_demo_path(memory_path)

avoid-aliased-function-imports


### Storing and retrieving rules

`store_knowledge` is the main write path: the agent provides the regex it created and the note that should be shown when it matches. `get_knowledge` can inspect all rules or filter by regex against the stored regex and note text.

In [ ]:
#| export
def _write_knowledge(data, path=None):
    pth = knowledge_path(path)
    pth.parent.mkdir(parents=True, exist_ok=True)
    pth.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")
    return pth

In [ ]:
#| export
def _check_regex(pattern):
    re.compile(pattern)
    return pattern

In [ ]:
#| export
def _print_json(value):
    print(json.dumps(value, indent=2, sort_keys=True))

In [ ]:
#| export
@call_parse
@tracked_call
def store_knowledge(
    apply_regex: str,  # Regex to apply to notebook code cells
    note: str,  # Behaviour note shown when the regex matches
    path: str | None = None,  # Override JSON memory path
):
    "Store or update one behaviour steering regex and note."
    _check_regex(apply_regex)
    data = _load_user_knowledge(path)
    now = time.time()
    item = {"regex": apply_regex, "note": note, "updated_ts": now}
    for idx, existing in enumerate(data["behaviours"]):
        if existing.get("regex") == apply_regex:
            item["created_ts"] = existing.get("created_ts", now)
            data["behaviours"][idx] = {**existing, **item}
            break
    else:
        item["created_ts"] = now
        data["behaviours"].append(item)
    pth = _write_knowledge(data, path)
    result = {"path": str(pth), "stored": item, "count": len(data["behaviours"])}
    _print_json(result)
    return cli_return(result)

In [ ]:
#| export
@call_parse
@tracked_call
def add_behaviour_steering(
    regex: str,  # Regex the agent created from a behaviour note
    path: str | None = None,  # Override JSON memory path
):
    "Add a behaviour steering regex with a generic note."
    return store_knowledge(regex, f"Behaviour steering matched: {regex}", path=path)

In [ ]:
#| export
@call_parse
@tracked_call
def get_knowledge(
    regex: str | None = None,  # Optional regex for filtering stored regexes and notes
    path: str | None = None,  # Override JSON memory path
    include_defaults: bool = True,  # Include built-in example behaviours
):
    "Return stored behaviour steering rules, optionally filtered by regex."
    data = load_knowledge(path, include_defaults=include_defaults)
    items = data.get("behaviours", [])
    if regex:
        query_re = re.compile(_check_regex(regex))
        items = [item for item in items if query_re.search(item.get("regex", "")) or query_re.search(item.get("note", ""))]
    result = {"path": str(knowledge_path(path)), "count": len(items), "behaviours": items}
    _print_json(result)
    return cli_return(result)

In [ ]:
memory_path = demo_path("knowledge-store.json")
try:
    stored = store_knowledge(r"from\s+\S+\s+import\s+\w+\s+as\s+\w+", "Do not import functions by alias unless needed.", path=str(memory_path))
    found = get_knowledge("alias", path=str(memory_path))
    print(found["behaviours"][-1]["note"])
    assert stored["count"] == 1
    assert found["count"] >= 1
finally:
    remove_demo_path(memory_path)

{
  "count": 1,
  "path": "nbs/data/knowledge-store.json",
  "stored": {
    "created_ts": 1779270447.670509,
    "note": "Do not import functions by alias unless needed.",
    "regex": "from\\s+\\S+\\s+import\\s+\\w+\\s+as\\s+\\w+",
    "updated_ts": 1779270447.670509
  }
}
{
  "behaviours": [
    {
      "id": "avoid-aliased-function-imports",
      "note": "Do not import functions by alias unless it is needed to avoid a real conflict.",
      "regex": "from\\s+\\S+\\s+import\\s+\\w+\\s+as\\s+\\w+",
      "source": "default"
    },
    {
      "created_ts": 1779270447.670509,
      "note": "Do not import functions by alias unless needed.",
      "regex": "from\\s+\\S+\\s+import\\s+\\w+\\s+as\\s+\\w+",
      "updated_ts": 1779270447.670509
    }
  ],
  "count": 2,
  "path": "nbs/data/knowledge-store.json"
}
Do not import functions by alias unless needed.


### Applying memory to notebooks

Style checks use the same matcher as MCP warnings. A rule is only useful if it points at a concrete notebook cell and line, so diagnostics include the path, cell id, line, note, regex, and matched text.

In [ ]:
#| export
def _is_notebook_path(path):
    path = Path(path)
    return path.suffix == ".ipynb" and ".ipynb_checkpoints" not in path.parts

In [ ]:
#| export
def _notebook_paths(path="."):
    raw = str(path)
    pth = Path(raw).expanduser()
    if any(char in raw for char in "*?[]"): candidates = [Path(item) for item in glob.glob(raw, recursive=True)]
    elif pth.is_dir(): candidates = list(pth.rglob("*.ipynb"))
    elif pth.is_file() and pth.suffix == ".ipynb": candidates = [pth]
    else: candidates = []
    return sorted({candidate for candidate in candidates if _is_notebook_path(candidate)})

In [ ]:
#| export
def _line_for_match(source, match):
    return source.count("\n", 0, match.start()) + 1

In [ ]:
#| export
def _knowledge_problem(nb_path, cell, line, item, match):
    note = item.get("note", "")
    return {
        "code": "knowledge-warning",
        "path": str(nb_path),
        "severity": "warning",
        "source": "nbskill-knowledge",
        "detail": "stored behaviour steering rule matched",
        "cell_id": getattr(cell, "id", ""),
        "line": line,
        "regex": item.get("regex", ""),
        "note": note,
        "match": match.group(0).splitlines()[0][:160],
    }

In [ ]:
#| export
def knowledge_style_problems(path=".", knowledge_path_override=None):
    "Return style diagnostics for stored behaviour steering regex matches."
    rules = load_knowledge(knowledge_path_override).get("behaviours", [])
    compiled = [(item, re.compile(item["regex"], re.MULTILINE)) for item in rules if item.get("regex")]
    if not compiled: return []
    problems = []
    for nb_path in _notebook_paths(path):
        try: nb = read_nb(nb_path)
        except FileNotFoundError: continue
        for cell in nb.cells:
            if getattr(cell, "cell_type", None) != "code": continue
            source = cell_source(cell)
            for item, pattern in compiled:
                for match in pattern.finditer(source):
                    problems.append(_knowledge_problem(nb_path, cell, _line_for_match(source, match), item, match))
    return problems

In [ ]:
from nbskill.foundation import write_demo_notebook
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import write_nb as _write_raw_nb

with write_demo_notebook("knowledge-match.json") as memory_path, write_demo_notebook("knowledge-demo.ipynb", reset=False) as demo_nb_path:
    store_knowledge(r"from\s+\S+\s+import\s+\w+\s+as\s+\w+", "Do not import functions by alias unless needed.", path=str(memory_path))
    _write_raw_nb(new_nb([mk_cell("from math import sqrt as square_root\nvalue = square_root(4)")]), demo_nb_path)
    problems = knowledge_style_problems(str(demo_nb_path), knowledge_path_override=str(memory_path))
    print(problems[0]["note"])
    print(len(problems),"problems found")
    assert len(problems)
    assert problems[0]["line"] == 1

{
  "count": 1,
  "path": "nbs/data/knowledge-match.json",
  "stored": {
    "created_ts": 1779270553.4094799,
    "note": "Do not import functions by alias unless needed.",
    "regex": "from\\s+\\S+\\s+import\\s+\\w+\\s+as\\s+\\w+",
    "updated_ts": 1779270553.4094799
  }
}
Do not import functions by alias unless it is needed to avoid a real conflict.
2 problems found


### How MCP loads the right memory

MCP read tools already know what the agent looked at: `nb_overview` has a notebook path, `nb_cell` has a notebook path and cell id, `show_doc` resolves a symbol to a cell, and edit tools know the cells they changed. The MCP response layer uses that scope to run `knowledge_style_problems` and then filters warnings down to those paths and cell ids.

That means the LLM does not need a separate retrieval step for normal work. When it reads a cell containing a stored pattern, the warning rides along with the tool result. `get_knowledge` is still useful when the agent wants to inspect or update the memory itself.